# Advanced models and comparator verification

This notebook inspects multivariate shared/independent segmentation, pooled replicate evidence, sliding-window approximation, supervised grouped classification, latent-template mixtures, and normalized comparator wrappers.

Each section states its estimand. Shared, independent, latent, and baseline outputs are fitted methods—not external truth. Controlled synthetic boundaries and labels are used only where explicitly declared.

In [ ]:
from __future__ import annotations

import json
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

import bayesbreak
from bayesbreak import (
    BayesBreakGaussian,
    BayesBreakGroupedClassifier,
    BayesBreakMixtureClassifier,
    IndependentMultivariateSegmenter,
    SharedBoundaryMultivariateSegmenter,
    SharedBoundaryReplicatesSegmenter,
    SlidingWindowSegmenter,
    run_dp_diagnostics,
)
from bayesbreak.baselines import available_algorithms, segment_with
from bayesbreak.metrics import boundary_metrics

SEED = 20260811
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
OUTPUT_DIR = ROOT / "results" / "notebook_verification" / "advanced_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

print({"python": platform.python_version(), "bayesbreak": bayesbreak.__version__, "seed": SEED})

## 1. Shared versus independent multivariate segmentation

Three channels share declared boundaries at 30 and 65 but differ in scale and noise. The shared model estimates one partition; the independent model estimates a partition per channel. Both are scored against the same declared boundaries.

In [ ]:
rng = np.random.default_rng(SEED)
n = 100
coordinates = np.arange(n, dtype=float).reshape(-1, 1)
truth = [30, 65]
channel_means = np.array([[0.0, 1.0, -1.0], [2.0, -1.0, 1.5], [-1.0, 0.5, 2.5]])
latent = np.vstack([
    np.tile(channel_means[0], (30, 1)),
    np.tile(channel_means[1], (35, 1)),
    np.tile(channel_means[2], (35, 1)),
])
noise_scales = np.array([0.25, 0.45, 0.7])
multivariate_y = latent + rng.normal(0.0, noise_scales, size=latent.shape)

started = time.perf_counter()
shared = SharedBoundaryMultivariateSegmenter(BayesBreakGaussian(k_max=6), k_max=6).fit(coordinates, multivariate_y)
shared_seconds = time.perf_counter() - started
started = time.perf_counter()
independent = IndependentMultivariateSegmenter(BayesBreakGaussian(k_max=6), k_max=6).fit(coordinates, multivariate_y)
independent_seconds = time.perf_counter() - started

shared_score = boundary_metrics(shared.map_boundaries_[1:-1], truth, tolerance=3, reference_type="simulated-truth", prediction_axis="index", reference_axis="index")
independent_scores = [boundary_metrics(model.map_boundaries_[1:-1], truth, tolerance=3, reference_type="simulated-truth", prediction_axis="index", reference_axis="index") for model in independent.channel_estimators_]
multivariate_frame = pd.DataFrame([
    {"method": "shared", "channel": "all", "boundaries": shared.map_boundaries_, "f1": shared_score.f1, "seconds": shared_seconds},
    *[{"method": "independent", "channel": index, "boundaries": model.map_boundaries_, "f1": score.f1, "seconds": independent_seconds / len(independent.channel_estimators_)} for index, (model, score) in enumerate(zip(independent.channel_estimators_, independent_scores, strict=True))],
])
display(multivariate_frame)
assert shared.map_curve_.shape == multivariate_y.shape
assert independent.map_curve_.shape == multivariate_y.shape

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for channel, ax in enumerate(axes):
    ax.scatter(coordinates[:, 0], multivariate_y[:, channel], s=10, alpha=0.45, color="#555555")
    ax.plot(coordinates[:, 0], shared.map_curve_[:, channel], color="#00798C", linewidth=2, label="shared MAP")
    ax.plot(coordinates[:, 0], independent.map_curve_[:, channel], color="#EDAE49", linewidth=1.5, linestyle=":", label="independent MAP")
    for boundary in truth:
        ax.axvline(boundary, color="#D1495B", linestyle="--", linewidth=1)
    ax.set(ylabel=f"channel {channel + 1}")
axes[0].legend(ncol=2)
axes[-1].set_xlabel("observation index")
fig.suptitle("Shared and channel-independent multivariate fits")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "multivariate_fits.png", bbox_inches="tight")
plt.show()

## 2. Verify pooled replicate evidence

For conditionally independent replicate sequences with common boundaries, pooled block log evidence must equal the sum of per-subject block log evidences when all fits use the same fixed hyperparameters.

In [ ]:
replicate_n = 60
replicate_x = np.arange(replicate_n).reshape(-1, 1)
replicate_latent = np.r_[np.zeros(25), np.full(20, 1.8), np.full(15, -0.8)]
replicates = [replicate_latent + rng.normal(0.0, 0.35, replicate_n) for _ in range(4)]
base_parameters = {"k_max": 5, "estimate_hyper": False, "nu": 0.0, "rho2": 10.0, "sigma2": 0.35**2}
replicate_model = SharedBoundaryReplicatesSegmenter(BayesBreakGaussian(**base_parameters)).fit(replicate_x, replicates)
individual_models = [BayesBreakGaussian(**base_parameters).fit(replicate_x, values) for values in replicates]
expected_pooled = sum(model.log_block_evidence_ for model in individual_models)
finite = np.isfinite(replicate_model.log_block_evidence_) & np.isfinite(expected_pooled)
max_pool_error = float(np.max(np.abs(replicate_model.log_block_evidence_[finite] - expected_pooled[finite])))
replicate_diagnostics = run_dp_diagnostics(replicate_model)
print({"k_map": replicate_model.k_map_, "boundaries": replicate_model.map_boundaries_, "max_pool_error": max_pool_error, "diagnostics": replicate_diagnostics.summary})
assert max_pool_error < 1e-9
assert replicate_diagnostics.passed

## 3. Inspect sliding-window approximation

The wrapper is exact only when the full sequence fits inside one window. On a long signal it stitches overlapping local fits, so this section checks finite outputs and recovery near a well-separated declared boundary without claiming global-DP equivalence.

In [ ]:
long_n = 240
long_x = np.arange(long_n).reshape(-1, 1)
long_y = np.r_[rng.normal(-1.0, 0.25, 120), rng.normal(2.0, 0.25, 120)]
started = time.perf_counter()
window_model = SlidingWindowSegmenter(BayesBreakGaussian(k_max=4), window_size=90, overlap=25).fit(long_x, long_y)
window_seconds = time.perf_counter() - started
interior_boundaries = [boundary for boundary in window_model.map_boundaries_ if 0 < boundary < long_n]
closest_window_error = min(abs(boundary - 120) for boundary in interior_boundaries)

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
axes[0].scatter(long_x[:, 0], long_y, s=9, alpha=0.45, color="#555555")
axes[0].plot(long_x[:, 0], window_model.map_curve_, color="#00798C", linewidth=2)
axes[0].axvline(120, color="#D1495B", linestyle="--", label="declared boundary")
axes[0].set(ylabel="response", title=f"Sliding-window MAP; nearest error={closest_window_error}")
axes[0].legend()
axes[1].fill_between(np.arange(1, long_n), window_model.boundary_marginals_, color="#30638E", alpha=0.55)
axes[1].axvline(120, color="#D1495B", linestyle="--")
axes[1].set(xlabel="index", ylabel="stitched marginal")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "sliding_window.png", bbox_inches="tight")
plt.show()
assert closest_window_error <= 8
assert np.all(np.isfinite(window_model.predict(long_x)))

## 4. Supervised groups and latent mixtures

Known-group classification uses declared labels. The mixture fit does not know those labels; its component indices are permutation-invariant, so evaluation aligns fitted components to declared groups with a maximum-overlap assignment. The aligned score is a controlled-fixture diagnostic, not proof of latent identifiability.

In [ ]:
sequence_n = 60
group_a = [np.r_[rng.normal(-0.5, 0.12, 30), rng.normal(0.5, 0.12, 30)] for _ in range(5)]
group_b = [np.r_[rng.normal(-4.0, 0.12, 30), rng.normal(4.0, 0.12, 30)] for _ in range(5)]
sequences = group_a + group_b
known_labels = np.array([0] * 5 + [1] * 5)
sequence_matrix = np.stack(sequences)

started = time.perf_counter()
grouped = BayesBreakGroupedClassifier(BayesBreakGaussian(k_max=4)).fit(sequences, known_labels)
grouped_seconds = time.perf_counter() - started
grouped_probability = grouped.predict_proba(sequences)
grouped_accuracy = float(np.mean(grouped.predict(sequences) == known_labels))

started = time.perf_counter()
mixture = BayesBreakMixtureClassifier(BayesBreakGaussian(k_max=4), n_groups=2, max_iter=8, tol=0.0, random_state=SEED).fit(sequence_matrix)
mixture_seconds = time.perf_counter() - started
mixture_probability = mixture.predict_proba(sequence_matrix)
raw_components = np.argmax(mixture_probability, axis=1)
contingency = np.zeros((2, 2), dtype=int)
for component, label in zip(raw_components, known_labels, strict=True):
    contingency[component, label] += 1
component_indices, label_indices = linear_sum_assignment(-contingency)
alignment = dict(zip(component_indices, label_indices, strict=True))
aligned_labels = np.array([alignment[component] for component in raw_components])
aligned_accuracy = float(np.mean(aligned_labels == known_labels))

classification_frame = pd.DataFrame(
    {
        "sequence": np.arange(len(sequences)),
        "declared_group": known_labels,
        "supervised_probability_group_1": grouped_probability[:, 1],
        "latent_component": raw_components,
        "aligned_latent_group": aligned_labels,
        "latent_max_probability": mixture_probability.max(axis=1),
    }
)
display(classification_frame)
print({"supervised_accuracy": grouped_accuracy, "aligned_latent_accuracy": aligned_accuracy, "objective": mixture.objective_})
assert np.allclose(grouped_probability.sum(axis=1), 1.0)
assert np.allclose(mixture_probability.sum(axis=1), 1.0)
assert np.all(np.isfinite(mixture.objective_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].imshow(mixture_probability.T, aspect="auto", cmap="viridis", vmin=0, vmax=1)
axes[0].set(title="Latent component responsibilities", xlabel="sequence", ylabel="component", yticks=[0, 1])
axes[0].axvline(4.5, color="white", linestyle="--", linewidth=1)
axes[1].plot(np.arange(1, len(mixture.objective_) + 1), mixture.objective_, marker="o", color="#00798C")
axes[1].set(title="Latent-mixture objective trace", xlabel="EM iteration", ylabel="objective")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "group_models.png", bbox_inches="tight")
plt.show()

## 5. Normalize and compare baseline outputs

Comparator wrappers return one schema regardless of upstream implementation. All methods below see the same clean three-regime signal and are scored against the same declared boundaries with the same tolerance. Method agreement is not labeled external truth.

In [ ]:
baseline_n = 90
baseline_truth = [30, 60]
baseline_y = np.r_[rng.normal(0.0, 0.2, 30), rng.normal(3.0, 0.2, 30), rng.normal(-2.0, 0.2, 30)]
baseline_specs = {
    "pelt": {"penalty": 10.0},
    "optimal_partitioning": {"n_bkps": 2},
    "binary_segmentation": {"n_bkps": 2},
    "wild_binary_segmentation": {"n_bkps": 2, "n_random_windows": 80, "random_state": SEED},
    "fearnhead_exact": {"k_max": 6, "geometric_rate": 0.3},
}
baseline_rows = []
for algorithm, parameters in baseline_specs.items():
    started = time.perf_counter()
    result = segment_with(algorithm, baseline_y, **parameters)
    elapsed = time.perf_counter() - started
    metrics = boundary_metrics(result.boundaries, baseline_truth, tolerance=3, reference_type="simulated-truth", prediction_axis="index", reference_axis="index")
    baseline_rows.append({"algorithm": algorithm, "package": result.package, "k": result.k, "boundaries": result.boundaries.tolist(), "f1": metrics.f1, "matched_mae": metrics.matched_mae, "runtime_ms": 1000 * elapsed})

bayes_started = time.perf_counter()
bayes_model = BayesBreakGaussian(k_max=6).fit(np.arange(baseline_n).reshape(-1, 1), baseline_y)
bayes_elapsed = time.perf_counter() - bayes_started
bayes_metrics = boundary_metrics(bayes_model.map_boundaries_[1:-1], baseline_truth, tolerance=3, reference_type="simulated-truth", prediction_axis="index", reference_axis="index")
baseline_rows.append({"algorithm": "bayesbreak_map", "package": f"bayesbreak {bayesbreak.__version__}", "k": bayes_model.k_map_, "boundaries": bayes_model.map_boundaries_[1:-1], "f1": bayes_metrics.f1, "matched_mae": bayes_metrics.matched_mae, "runtime_ms": 1000 * bayes_elapsed})
baseline_frame = pd.DataFrame(baseline_rows)
display(baseline_frame)
print("Available registry entries:", available_algorithms())
assert (baseline_frame.f1 >= 0).all() and (baseline_frame.f1 <= 1).all()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(baseline_frame.algorithm, baseline_frame.f1, color="#00798C")
axes[0].set(title="Boundary F1 at tolerance 3", xlabel="F1", xlim=(0, 1.05))
axes[1].barh(baseline_frame.algorithm, baseline_frame.runtime_ms, color="#EDAE49")
axes[1].set(title="Observed comparator runtime", xlabel="milliseconds")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "baseline_comparison.png", bbox_inches="tight")
plt.show()

## 6. Generate the advanced-model report

The report records structural invariants, controlled-fixture metrics, and local timings. It does not collapse methods into a single ranking because they target different modeling assumptions.

In [ ]:
multivariate_frame.to_json(OUTPUT_DIR / "multivariate_results.json", orient="records", indent=2)
classification_frame.to_csv(OUTPUT_DIR / "classification_results.csv", index=False)
baseline_frame.to_json(OUTPUT_DIR / "baseline_results.json", orient="records", indent=2)
report = {
    "bayesbreak_version": bayesbreak.__version__,
    "seed": SEED,
    "multivariate": {"shared_f1": shared_score.f1, "independent_mean_f1": float(np.mean([score.f1 for score in independent_scores])), "shared_seconds": shared_seconds, "independent_seconds": independent_seconds},
    "replicates": {"pooled_max_abs_error": max_pool_error, "diagnostics_passed": replicate_diagnostics.passed},
    "sliding_window": {"nearest_boundary_error": closest_window_error, "seconds": window_seconds, "interpretation": "approximate stitched fit"},
    "groups": {"supervised_accuracy": grouped_accuracy, "aligned_latent_accuracy": aligned_accuracy, "supervised_seconds": grouped_seconds, "mixture_seconds": mixture_seconds},
    "baselines": {"algorithms": baseline_frame.algorithm.tolist(), "all_metrics_finite": bool(np.isfinite(baseline_frame[["f1", "runtime_ms"]]).all().all())},
}
report["all_required_checks_passed"] = bool(max_pool_error < 1e-9 and replicate_diagnostics.passed and closest_window_error <= 8 and report["baselines"]["all_metrics_finite"])
(OUTPUT_DIR / "report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
assert report["all_required_checks_passed"]